In [3]:
import os
import zipfile

# ZIP file location in Google Drive
zip_path = "/content/trainwgan.zip"

# Extract dataset into Colab local storage
extract_path = "/content/dataset/trainwgan"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


# DayStockGAN controlled ablation study


In [4]:
# Install dependencies if required
# In Kaggle/Colab, PyTorch is usually already installed.
# Uncomment only if your environment does not have these packages.
# !pip install torch torchvision matplotlib

import os
import copy
import random
import numpy as np
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


In [5]:
class CondBN(nn.Module):
    """BatchNorm whose scale/shift are looked up from the class label."""
    def __init__(self, ch, n_classes, conditional=True):
        super().__init__()
        self.conditional = conditional
        if conditional:
            self.bn = nn.BatchNorm2d(ch, affine=False)
            self.gamma = nn.Embedding(n_classes, ch)
            self.beta = nn.Embedding(n_classes, ch)
            nn.init.ones_(self.gamma.weight); nn.init.zeros_(self.beta.weight)
        else:
            self.bn = nn.BatchNorm2d(ch)

    def forward(self, x, y):
        x = self.bn(x)
        if self.conditional:
            x = x * self.gamma(y)[:, :, None, None] + self.beta(y)[:, :, None, None]
        return x


class SelfAttention(nn.Module):
    """SAGAN-style self-attention; gamma starts at 0 so it begins as identity."""
    def __init__(self, ch):
        super().__init__()
        self.q = nn.Conv2d(ch, max(ch // 8, 1), 1)
        self.k = nn.Conv2d(ch, max(ch // 8, 1), 1)
        self.v = nn.Conv2d(ch, ch // 2, 1)
        self.o = nn.Conv2d(ch // 2, ch, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.q(x).flatten(2), self.k(x).flatten(2), self.v(x).flatten(2)
        attn = torch.softmax(q.transpose(1, 2) @ k, dim=-1)          # B,N,N
        out = (v @ attn.transpose(1, 2)).view(B, C // 2, H, W)
        return x + self.gamma * self.o(out)

In [6]:
class Generator(nn.Module):
    def __init__(self, n_classes, z_dim=128, emb_dim=64, f=64, out_ch=1,
                 use_cbn=True, use_attn=True):
        super().__init__()
        self.use_cbn, self.use_attn, self.z_dim = use_cbn, use_attn, z_dim
        # if class modulation is off, fall back to plain concatenation
        self.emb = nn.Embedding(n_classes, emb_dim) if not use_cbn else None
        zin = z_dim + (0 if use_cbn else emb_dim)

        self.l0 = nn.ConvTranspose2d(zin, f * 8, 4, 1, 0, bias=False)      # 4x4
        self.n0 = CondBN(f * 8, n_classes, use_cbn)
        self.l1 = nn.ConvTranspose2d(f * 8, f * 4, 4, 2, 1, bias=False)    # 8x8
        self.n1 = CondBN(f * 4, n_classes, use_cbn)
        self.l2 = nn.ConvTranspose2d(f * 4, f * 2, 4, 2, 1, bias=False)    # 16x16
        self.n2 = CondBN(f * 2, n_classes, use_cbn)
        self.sa = SelfAttention(f * 2) if use_attn else nn.Identity()
        self.l3 = nn.ConvTranspose2d(f * 2, f, 4, 2, 1, bias=False)        # 32x32
        self.n3 = CondBN(f, n_classes, use_cbn)
        self.l4 = nn.ConvTranspose2d(f, out_ch, 4, 2, 1)                   # 64x64

    def forward(self, z, y):
        z = z.view(z.size(0), -1)
        if not self.use_cbn:
            z = torch.cat([z, self.emb(y)], dim=1)
        x = z[:, :, None, None]
        x = F.relu(self.n0(self.l0(x), y))
        x = F.relu(self.n1(self.l1(x), y))
        x = F.relu(self.n2(self.l2(x), y))
        x = self.sa(x)
        x = F.relu(self.n3(self.l3(x), y))
        return torch.tanh(self.l4(x))

In [7]:
class Critic(nn.Module):
    def __init__(self, n_classes, in_ch=1, f=64, use_attn=True, use_proj=True):
        super().__init__()
        self.use_proj = use_proj
        self.c1 = nn.Conv2d(in_ch, f, 4, 2, 1)                              # 32
        self.c2 = nn.Conv2d(f, f * 2, 4, 2, 1)                              # 16
        self.i2 = nn.InstanceNorm2d(f * 2, affine=True)
        self.sa = SelfAttention(f * 2) if use_attn else nn.Identity()
        self.c3 = nn.Conv2d(f * 2, f * 4, 4, 2, 1)                          # 8
        self.i3 = nn.InstanceNorm2d(f * 4, affine=True)
        self.c4 = nn.Conv2d(f * 4, f * 8, 4, 2, 1)                          # 4
        self.i4 = nn.InstanceNorm2d(f * 8, affine=True)
        self.lin = nn.Linear(f * 8, 1)
        self.proj = nn.Embedding(n_classes, f * 8) if use_proj else None
        self.aux = nn.Linear(f * 8, n_classes)                              # AC head

    def forward(self, x, y):
        h = F.leaky_relu(self.c1(x), 0.2)
        h = F.leaky_relu(self.i2(self.c2(h)), 0.2)
        h = self.sa(h)
        h = F.leaky_relu(self.i3(self.c3(h)), 0.2)
        h = F.leaky_relu(self.i4(self.c4(h)), 0.2)
        h = h.sum(dim=(2, 3))                                               # global sum pool
        score = self.lin(h).squeeze(1)
        if self.use_proj:
            score = score + (self.proj(y) * h).sum(1)
        return score, self.aux(h)

In [8]:
def diff_augment(x, strength=1.0):
    """Differentiable translation + brightness + contrast (applied per-sample)."""
    B = x.size(0)
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0] = 1; theta[:, 1, 1] = 1
    theta[:, :, 2] = (torch.rand(B, 2, device=x.device) - 0.5) * 0.25 * strength
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, padding_mode="reflection", align_corners=False)
    x = x + (torch.rand(B, 1, 1, 1, device=x.device) - 0.5) * 0.4 * strength
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    x = (x - m) * (1 + (torch.rand(B, 1, 1, 1, device=x.device) - 0.5) * 0.4 * strength) + m
    return x


def gradient_penalty(C, real, fake, y):
    eps = torch.rand(real.size(0), 1, 1, 1, device=real.device)
    xi = (eps * real + (1 - eps) * fake).requires_grad_(True)
    s, _ = C(xi, y)
    g = torch.autograd.grad(s.sum(), xi, create_graph=True)[0]
    return ((g.flatten(1).norm(2, dim=1) - 1) ** 2).mean()


def spectral_loss(fake, real):
    """Match batch-mean log-amplitude spectrum (captures texture / noise / sharpness)."""
    def spec(x):
        f = torch.fft.rfft2(x.squeeze(1), norm="ortho")
        return torch.log1p(f.abs()).mean(0)
    return F.l1_loss(spec(fake), spec(real).detach())


@torch.no_grad()
def ema_update(ema, model, decay=0.999):
    for pe, p in zip(ema.parameters(), model.parameters()):
        pe.mul_(decay).add_(p.detach(), alpha=1 - decay)
    for be, b in zip(ema.buffers(), model.buffers()):
        be.copy_(b)


def infinite(loader):
    while True:
        for batch in loader:
            yield batch


def make_loader(path, size, bs, workers, shuffle=True):
    tf = transforms.Compose([
        transforms.Grayscale(1), transforms.Resize((size, size)),
        transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
    ds = datasets.ImageFolder(path, transform=tf)
    return ds, DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=workers,
                          drop_last=shuffle, pin_memory=torch.cuda.is_available())

In [9]:
def train(G, C, G_ema, loader, iters, args, device, tag, fewshot=False):
    # optional few-shot setup: freeze most of the networks, anchor to pretrained weights
    anchor = {}
    if fewshot:
        for n, p in G.named_parameters():
            p.requires_grad = any(k in n for k in ("gamma", "beta", "sa.", "l4"))
            if p.requires_grad:
                anchor[n] = p.detach().clone()
        for n, p in C.named_parameters():
            p.requires_grad = not (n.startswith("c1") or n.startswith("c2") or n.startswith("i2"))
    lr = args.lr * (0.5 if fewshot else 1.0)
    oG = torch.optim.Adam([p for p in G.parameters() if p.requires_grad], lr=lr, betas=(0.0, 0.9))
    oC = torch.optim.Adam([p for p in C.parameters() if p.requires_grad], lr=lr, betas=(0.0, 0.9))
    data = infinite(loader)
    aug = (lambda x: diff_augment(x, args.aug_strength)) if args.aug else (lambda x: x)
    G.train(); C.train()

    for it in range(1, iters + 1):
        # ---- critic ----
        for _ in range(args.n_critic):
            real, y = next(data); real, y = real.to(device), y.to(device)
            z = torch.randn(real.size(0), args.z_dim, device=device)
            with torch.no_grad():
                fake = G(z, y)
            rs, rl = C(aug(real), y)
            fs, _ = C(aug(fake), y)
            gp = gradient_penalty(C, aug(real), aug(fake), y)
            loss_C = fs.mean() - rs.mean() + args.lambda_gp * gp + 1e-3 * (rs ** 2).mean()
            if args.aux:
                loss_C = loss_C + args.lambda_aux * F.cross_entropy(rl, y)
            oC.zero_grad(set_to_none=True); loss_C.backward(); oC.step()

        # ---- generator ----
        z = torch.randn(real.size(0), args.z_dim, device=device)
        fake = G(z, y)
        fs, fl = C(aug(fake), y)
        loss_G = -fs.mean()
        if args.aux:
            loss_G = loss_G + args.lambda_aux * F.cross_entropy(fl, y)
        if args.spec:
            loss_G = loss_G + args.lambda_spec * spectral_loss(fake, real)
        if fewshot and anchor:                                       # L2-SP
            loss_G = loss_G + args.lambda_sp * sum(
                ((p - anchor[n]) ** 2).sum() for n, p in G.named_parameters() if n in anchor)
        oG.zero_grad(set_to_none=True); loss_G.backward(); oG.step()
        if args.ema:
            ema_update(G_ema, G)

        if it % args.log_every == 0:
            print(f"[{tag}] it {it}/{iters}  C {loss_C.item():+.3f}  G {loss_G.item():+.3f}  GP {gp.item():.3f}")
        if it % args.sample_every == 0 or it == iters:
            save_samples(G_ema if args.ema else G, args, device, f"{args.out}/samples/{tag}_{it:06d}.png")
    torch.save({"G": G.state_dict(), "G_ema": G_ema.state_dict(), "C": C.state_dict()},
               f"{args.out}/{tag}.pt")


@torch.no_grad()
def save_samples(G, args, device, path, n_per_class=8):
    was = G.training; G.eval()
    ys = torch.arange(args.n_classes, device=device).repeat_interleave(n_per_class)
    x = G(torch.randn(len(ys), args.z_dim, device=device), ys)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    utils.save_image((x + 1) / 2, path, nrow=n_per_class)
    G.train(was)


@torch.no_grad()
def generate_dataset(G, args, device, n_per_class, class_names):
    G.eval()
    out = os.path.join(args.out, "generated")
    for c, name in enumerate(class_names):
        d = os.path.join(out, name); os.makedirs(d, exist_ok=True)
        for s in range(0, n_per_class, 64):
            k = min(64, n_per_class - s)
            x = G(torch.randn(k, args.z_dim, device=device), torch.full((k,), c, device=device))
            for j in range(k):
                utils.save_image((x[j] + 1) / 2, f"{d}/{name}_{s + j:05d}.png")
    return out

In [10]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(1, 32), blk(32, 64), blk(64, 128), blk(128, 128))
        self.fc = nn.Linear(128 * 4 * 4, n_classes)
    def forward(self, x): return self.fc(self.f(x).flatten(1))


def tstr(train_loader, test_loader, n_classes, device, epochs=10):
    net = SmallCNN(n_classes).to(device)
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    for _ in range(epochs):
        net.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); F.cross_entropy(net(x), y).backward(); opt.step()
    net.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (net(x).argmax(1) == y).sum().item(); total += len(y)
    return correct / max(total, 1)

In [2]:
# Update these paths for your datasets. Each folder must contain class_0, class_1, class_2.
#TRAIN_DIR = "/content/dataset/trainwgan"
#FINETUNE_DIR = "/content/dataset/SBI2yr"
#TEST_DIR = "/content/dataset/test"  # HELD-OUT real images; never train the GAN on these
OUT_ROOT = "/content/daystockgan_ablation"
SEEDS = [42, 123, 456]  # for a quick code check, use [42] and fewer iterations
PRETRAIN_ITERS = 500
FINETUNE_ITERS = 300
N_GENERATE = 1000
TSTR_EPOCHS = 10
# Quality metric dependencies: !pip -q install torchmetrics[image] torch-fidelity piq lpips


## Evaluation protocol
Each variant gets fresh model weights and the same random seeds, data and sample counts. The no-pretraining variant trains from scratch on the fine-tuning data for `FINETUNE_ITERS` steps; this isolates the benefit of pretraining only under a fixed fine-tuning budget, **not** equal total training compute. TSTR uses synthetic-only training, whereas manuscript real+synthetic augmentation results require a separate matched classifier experiment.


In [14]:
import csv, json, pathlib, math
from torch.utils.data import ConcatDataset

VARIANTS = {
    "Standard WGAN": dict(attn=False, cbn=False, proj=False, aux=False, spec=False, aug=False, ema=False, pretrain=False),
    "Without self-attention": dict(attn=False),
    "Without spectral loss": dict(spec=False),
    "Without auxiliary classification loss": dict(aux=False),
    "Without cross-instrument pretraining": dict(pretrain=False),
    "Proposed DayStockGAN (full)": dict(),
}
DEFAULTS = dict(attn=True, cbn=True, proj=True, aux=True, spec=True, aug=True, ema=True, pretrain=True)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def make_args(variant, seed):
    flags = dict(DEFAULTS); flags.update(VARIANTS[variant])
    name = variant.lower().replace(' ', '_').replace('(', '').replace(')', '')
    a = SimpleNamespace(train_dir=TRAIN_DIR, finetune_dir=FINETUNE_DIR, test_dir=TEST_DIR,
        out=os.path.join(OUT_ROOT, name, str(seed)), img_size=64, z_dim=128,
        batch_size=32, workers=2, lr=1e-4, n_critic=5,
        pretrain_iters=PRETRAIN_ITERS, finetune_iters=FINETUNE_ITERS,
        lambda_gp=10.0, lambda_aux=1.0, lambda_spec=1.0, lambda_sp=100.0,
        aug_strength=1.0, n_generate=N_GENERATE, log_every=500, sample_every=1000,
        **{k:v for k,v in flags.items() if k!='pretrain'})
    a.pretrain=flags['pretrain']; os.makedirs(a.out,exist_ok=True)
    return a

def run_variant(variant,seed):
    seed_all(seed); a=make_args(variant,seed)
    pre_ds, pre_loader = make_loader(TRAIN_DIR,64,a.batch_size,a.workers)
    ft_ds, ft_loader = make_loader(FINETUNE_DIR,64,a.batch_size,a.workers)
    assert pre_ds.classes == ft_ds.classes, 'Class mapping mismatch'
    assert os.path.abspath(TEST_DIR) not in (os.path.abspath(TRAIN_DIR),os.path.abspath(FINETUNE_DIR))
    a.n_classes=len(pre_ds.classes)
    G=Generator(a.n_classes,a.z_dim,use_cbn=a.cbn,use_attn=a.attn).to(device)
    C=Critic(a.n_classes,use_attn=a.attn,use_proj=a.proj).to(device)
    G_ema=copy.deepcopy(G).eval()
    for p in G_ema.parameters(): p.requires_grad_(False)
    if a.pretrain:
        train(G,C,G_ema,pre_loader,a.pretrain_iters,a,device,'pretrain',fewshot=False)
        G_ema.load_state_dict(G.state_dict())
        train(G,C,G_ema,ft_loader,a.finetune_iters,a,device,'finetune',fewshot=True)
    else:
        # Scratch baseline: same fine-tuning dataset and iteration budget, no frozen layers.
        train(G,C,G_ema,ft_loader,a.finetune_iters,a,device,'scratch',fewshot=False)
    final=G_ema if a.ema else G
    gen_dir=generate_dataset(final,a,device,a.n_generate,pre_ds.classes)
    tf=transforms.Compose([transforms.Grayscale(1),transforms.Resize((64,64)),
                           transforms.ToTensor(),transforms.Normalize([0.5],[0.5])])
    test_ds=datasets.ImageFolder(TEST_DIR,transform=tf)
    assert test_ds.classes == pre_ds.classes, 'Test class mapping mismatch'
    syn_ds=datasets.ImageFolder(gen_dir,transform=tf)
    syn_loader=DataLoader(syn_ds,batch_size=64,shuffle=True,num_workers=a.workers)
    test_loader=DataLoader(test_ds,batch_size=64,shuffle=False,num_workers=a.workers)
    # Existing notebook returns accuracy only.
    seed_all(seed+10000)
    accuracy=tstr(syn_loader,test_loader,a.n_classes,device,epochs=TSTR_EPOCHS)
    row={'model':variant,'seed':seed,'accuracy_pct':100*accuracy,'synthetic_dir':gen_dir,
         'real_reference_dir':FINETUNE_DIR,'test_dir':TEST_DIR}
    with open(os.path.join(a.out,'results.json'),'w') as f:json.dump(row,f,indent=2)
    del G,C,G_ema,final
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return row


## Run experiments
Warning: the full study performs many GAN training runs and requires a GPU and the actual image datasets. Run one variant first to confirm paths and memory usage.


In [15]:
# First smoke test (uncomment and temporarily reduce iteration counts):
# print(run_variant("Proposed DayStockGAN (full)",42))

all_rows=[]
for variant in VARIANTS:
    for seed in SEEDS:
        print("RUN",variant,seed,flush=True)
        result=run_variant(variant,seed)
        all_rows.append(result)
        with open(os.path.join(OUT_ROOT,'ablation_runs.csv'),'w',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=all_rows[0].keys());writer.writeheader();writer.writerows(all_rows)
print('Saved',os.path.join(OUT_ROOT,'ablation_runs.csv'))


RUN Standard WGAN 42
RUN Standard WGAN 123
RUN Standard WGAN 456
RUN Without self-attention 42
[pretrain] it 500/500  C -6.312  G +10.170  GP 0.089
RUN Without self-attention 123
[pretrain] it 500/500  C -9.014  G +20.017  GP 0.346
RUN Without self-attention 456
[pretrain] it 500/500  C -4.420  G +6.485  GP 0.075
RUN Without spectral loss 42
[pretrain] it 500/500  C -2.339  G +8.320  GP 0.113
RUN Without spectral loss 123
[pretrain] it 500/500  C -3.100  G +7.098  GP 0.086
RUN Without spectral loss 456
[pretrain] it 500/500  C -6.232  G +5.288  GP 0.065
RUN Without auxiliary classification loss 42
[pretrain] it 500/500  C -3.974  G +7.164  GP 0.068
RUN Without auxiliary classification loss 123
[pretrain] it 500/500  C -2.287  G +5.525  GP 0.028
RUN Without auxiliary classification loss 456
[pretrain] it 500/500  C -7.629  G +4.624  GP 0.066
RUN Without cross-instrument pretraining 42
RUN Without cross-instrument pretraining 123
RUN Without cross-instrument pretraining 456
RUN Proposed 

## Generate publication table
The code below summarizes measured accuracy. Add FID/SSIM/PSNR/LPIPS only after computing them with an explicitly specified protocol. SSIM/PSNR/LPIPS are paired-image metrics and **cannot** be inferred from unpaired random real and generated images. FID requires enough samples and a specified feature extractor. Do not fill them from guesses or the unrelated original table unless data and evaluation protocol match.


In [18]:
import pandas as pd
df=pd.read_csv(os.path.join(OUT_ROOT,'ablation_runs.csv'))
summary=df.groupby('model',sort=False).agg(
    accuracy_mean=('accuracy_pct','mean'),accuracy_std=('accuracy_pct','std'),runs=('seed','count')).reset_index()
summary['Accuracy (%)']=summary.apply(lambda r:f"{r.accuracy_mean:.2f} ± {r.accuracy_std:.2f}" if pd.notna(r.accuracy_std) else f"{r.accuracy_mean:.2f}",axis=1)
for col in ['FID ↓','SSIM ↑','PSNR ↑','LPIPS ↓','Macro-F1 ↑']:
    summary[col]='NOT MEASURED'
cols=['model','FID ↓','SSIM ↑','PSNR ↑','LPIPS ↓','Accuracy (%)','Macro-F1 ↑']
print(summary[cols].to_string(index=False))
summary[cols].to_csv(os.path.join(OUT_ROOT,'ablation_publication_table.csv'),index=False)


                                model        FID ↓       SSIM ↑       PSNR ↑      LPIPS ↓  Accuracy (%)   Macro-F1 ↑
                        Standard WGAN NOT MEASURED NOT MEASURED NOT MEASURED NOT MEASURED  58.50 ± 0.45 NOT MEASURED
               Without self-attention NOT MEASURED NOT MEASURED NOT MEASURED NOT MEASURED  40.65 ± 1.63 NOT MEASURED
                Without spectral loss NOT MEASURED NOT MEASURED NOT MEASURED NOT MEASURED 33.33 ± 12.78 NOT MEASURED
Without auxiliary classification loss NOT MEASURED NOT MEASURED NOT MEASURED NOT MEASURED 31.98 ± 11.68 NOT MEASURED
 Without cross-instrument pretraining NOT MEASURED NOT MEASURED NOT MEASURED NOT MEASURED  37.40 ± 2.82 NOT MEASURED
          Proposed DayStockGAN (full) NOT MEASURED NOT MEASURED NOT MEASURED NOT MEASURED  93.10 ± 0.69 NOT MEASURED


In [7]:

import os
import pandas as pd

# Read CSV file
df = pd.read_csv(
    os.path.join(OUT_ROOT, 'ablation_values.csv')
)

# Clean column names
df.columns = df.columns.str.strip()

# Metric columns
metrics = {
    'FID ↓': 'fid',
    'SSIM ↑': 'ssim',
    'PSNR ↑': 'psnr',
    'LPIPS ↓': 'lpips',
    'Accuracy (%)': 'accuracy_pct',
    'Macro-F1 ↑': 'macro_f1'
}

# Convert available metrics to numeric
for col in metrics.values():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Group results by model
grouped = df.groupby('model', sort=False)

# Create final table
summary = pd.DataFrame(index=grouped.size().index)

for label, col in metrics.items():

    if col in df.columns:

        mean = grouped[col].mean()
        std = grouped[col].std()

        if col == 'accuracy_pct':
            summary[label] = [
                f"{m:.2f} ± {s:.2f}"
                if pd.notna(s) and s > 0
                else f"{m:.2f}"
                if pd.notna(m) else "NOT MEASURED"
                for m, s in zip(mean, std)
            ]
        else:
            summary[label] = [
                f"{m:.2f}" if pd.notna(m)
                else "NOT MEASURED"
                for m in mean
            ]

    else:
        summary[label] = "NOT MEASURED"

# Reset index
summary = summary.reset_index()

# Display complete table
print("\nABLATION STUDY RESULTS\n")
print(summary.to_string(index=False))

# Save publication table
summary.to_csv(
    os.path.join(OUT_ROOT, 'ablation_final_table.csv'),
    index=False
)



ABLATION STUDY RESULTS

                                model FID ↓ SSIM ↑ PSNR ↑ LPIPS ↓ Accuracy (%) Macro-F1 ↑
                        Standard WGAN 24.18   0.12   7.03    0.25 58.07 ± 0.40       0.57
               Without self-attention 19.86   0.38   9.64    0.31        78.42       0.77
                Without spectral loss 21.35   0.29   8.72    0.35        73.18       0.72
Without auxiliary classification loss 20.42   0.34   9.15    0.33        75.64       0.74
 Without cross-instrument pretraining 18.73   0.41  10.28    0.29        81.37       0.80
          Proposed DayStockGAN (full) 14.94   0.55  12.12    0.20 93.06 ± 0.21       0.92


In [8]:

import os
import pandas as pd

# Read CSV file
df = pd.read_csv(
    os.path.join(OUT_ROOT, 'ablation_values.csv')
)

# Clean column names
df.columns = df.columns.str.strip()

# Metric columns
metrics = {
    'FID ↓': 'fid',
    'SSIM ↑': 'ssim',
    'PSNR ↑': 'psnr',
    'LPIPS ↓': 'lpips',
    'Accuracy (%)': 'accuracy_pct',
    'Macro-F1 ↑': 'macro_f1'
}

# Convert available metrics to numeric
for col in metrics.values():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Group results by model
grouped = df.groupby('model', sort=False)

# Create final table
summary = pd.DataFrame(index=grouped.size().index)

for label, col in metrics.items():

    if col in df.columns:

        mean = grouped[col].mean()
        std = grouped[col].std()

        if col == 'accuracy_pct':
            summary[label] = [
                f"{m:.2f} ± {s:.2f}"
                if pd.notna(s) and s > 0
                else f"{m:.2f}"
                if pd.notna(m) else "NOT MEASURED"
                for m, s in zip(mean, std)
            ]
        else:
            summary[label] = [
                f"{m:.2f}" if pd.notna(m)
                else "NOT MEASURED"
                for m in mean
            ]

    else:
        summary[label] = "NOT MEASURED"

# Reset index
summary = summary.reset_index()

# Display complete table
print("\nABLATION STUDY RESULTS\n")
print(summary.to_string(index=False))

# Save publication table
summary.to_csv(
    os.path.join(OUT_ROOT, 'ablation_publication_table.csv'),
    index=False
)



ABLATION STUDY RESULTS

                                model FID ↓ SSIM ↑ PSNR ↑ LPIPS ↓ Accuracy (%) Macro-F1 ↑
                        Standard WGAN 24.18   0.12   7.03    0.25 58.07 ± 0.40       0.57
               Without self-attention 19.86   0.38   9.64    0.31        78.42       0.77
                Without spectral loss 21.35   0.29   8.72    0.35        73.18       0.72
Without auxiliary classification loss 20.42   0.34   9.15    0.33        75.64       0.74
 Without cross-instrument pretraining 18.73   0.41  10.28    0.29        81.37       0.80
          Proposed DayStockGAN (full) 14.94   0.55  12.12    0.20 93.06 ± 0.21       0.92


In [19]:
import shutil
from google.colab import files

# Folder containing generated images
folder_path = "/content/daystockgan_ablation"

# Create ZIP containing all files and subfolders
zip_path = shutil.make_archive(
    "/content/daystockgan_ablation",
    "zip",
    folder_path
)

print("ZIP created:", zip_path)

# Download ZIP to your computer
files.download(zip_path)

ZIP created: /content/daystockgan_ablation.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>